In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU:", torch.cuda.get_device_name(0))
    print("GPU VRAM Total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("Running on CPU only. We need to install CUDA-enabled PyTorch.")

CUDA Available: True
Active GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU VRAM Total: 8.59 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!unzip "/content/drive/MyDrive/BGL Log/BGL.log.zip" -d /content/

Mounted at /content/drive
Archive:  /content/drive/MyDrive/BGL Log/BGL.log.zip
  inflating: /content/BGL.log        


In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
# ==========================================
# CELL 1: REPRODUCIBILITY, PARSING & ENHANCED EXTRACTION
# ==========================================
import random
import numpy as np
import torch
import pandas as pd
import re
import time
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

file_path = 'BGL.log'

bgl_pattern = re.compile(
    r'^(?P<Label>\S+)\s+(?P<Timestamp>\d+)\s+(?P<Date>\d{4}\.\d{2}\.\d{2})\s+'
    r'(?P<Node>\S+)\s+(?P<Time>\d{4}-\d{2}-\d{2}-\d{2}\.\d{2}\.\d{2}\.\d+)\s+'
    r'(?P<NodeRepeat>\S+)\s+(?P<Type>\S+)\s+(?P<Component>\S+)\s+'
    r'(?P<Level>\S+)\s+(?P<Content>.*)$'
)

parsed_logs = []
start_time = time.time()

print("Beginning parsing of massive dataset...")
with open(file_path, 'r') as file:
    for i, line in enumerate(file, start=1):
        match = bgl_pattern.match(line.strip())
        if match:
            parsed_logs.append(match.groupdict())
        if i % 500000 == 0:
            print(f"Parsed {i:,} lines...")

df = pd.DataFrame(parsed_logs)
df['Is_Anomaly'] = df['Label'].apply(lambda x: 0 if x == '-' else 1)

# Enhanced Regex-Based Event Template Extraction
def extract_bgl_template(content):
    t = re.sub(r'0x[a-fA-F0-9]+', '<HEX>', content)
    t = re.sub(r'\d+\.\d+\.\d+\.\d+', '<IP>', t)
    t = re.sub(r'[A-Z0-9]+-[A-Z0-9]+-[A-Z0-9]+-[A-Z0-9]+:[A-Z0-9]+-[A-Z0-9]+', '<NODE>', t)
    t = re.sub(r'/[a-zA-Z0-9_./-]+', '<PATH>', t)
    t = re.sub(r'(?i)(mid|core|tree|torus)\s+\d+', r'\1 <NUM>', t)
    t = re.sub(r'\b\d+\b', '<NUM>', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['Event template'] = df['Content'].apply(extract_bgl_template)
df['Timestamp'] = pd.to_numeric(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f"Parsing Complete: {len(df):,} total logs in {round(time.time() - start_time, 2)} seconds.")

print("\n--- LOG-LEVEL CLASS DISTRIBUTION ---")
print(df['Is_Anomaly'].value_counts())
print(df['Is_Anomaly'].value_counts(normalize=True))

Beginning parsing of massive dataset...
Parsed 500,000 lines...
Parsed 1,000,000 lines...
Parsed 1,500,000 lines...
Parsed 2,000,000 lines...
Parsed 2,500,000 lines...
Parsed 3,000,000 lines...
Parsed 3,500,000 lines...
Parsed 4,000,000 lines...
Parsed 4,500,000 lines...
Parsing Complete: 4,713,493 total logs in 118.53 seconds.

--- LOG-LEVEL CLASS DISTRIBUTION ---
Is_Anomaly
0    4365033
1     348460
Name: count, dtype: int64
Is_Anomaly
0    0.926072
1    0.073928
Name: proportion, dtype: float64


In [ ]:
# ==========================================
# CELL 2: SPLITTING, VOCABULARY & WINDOW GENERATION
# ==========================================
n_total = len(df)
train_idx = int(n_total * 0.7)
val_idx = int(n_total * 0.8)

train_df = df.iloc[:train_idx].copy()
val_df = df.iloc[train_idx:val_idx].copy()
test_df = df.iloc[val_idx:].copy()

print(f"\nChronological Split - Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,} logs.")

unique_templates = train_df['Event template'].unique()
template_to_id = {template: idx + 1 for idx, template in enumerate(unique_templates)}

def map_tokens(data):
    return data['Event template'].map(template_to_id).fillna(0).astype(int)

train_df['Event_Token'] = map_tokens(train_df)
val_df['Event_Token'] = map_tokens(val_df)
test_df['Event_Token'] = map_tokens(test_df)

def create_windows(data_subset, window_size=20, step_size=5):
    tokens = data_subset['Event_Token'].values
    anomaly_flags = data_subset['Is_Anomaly'].values
    levels = data_subset['Level'].values

    sequences, labels, severity_scores = [], [], []
    critical_levels = ['WARNING', 'ERROR', 'SEVERE', 'FATAL', 'FAILURE']

    for i in range(0, len(tokens) - window_size + 1, step_size):
        seq = tokens[i:i + window_size]
        if len(seq) < window_size:
            continue
        window_is_anomaly = int(anomaly_flags[i:i + window_size].sum() > 0)
        severity_score = sum(level in critical_levels for level in levels[i:i + window_size])

        sequences.append(seq)
        labels.append(window_is_anomaly)
        severity_scores.append(severity_score)

    return np.array(sequences), np.array(labels), np.array(severity_scores)

# Overlapping Windows (Primary)
X_train, y_train, train_severity = create_windows(train_df, step_size=5)
X_val, y_val, val_severity = create_windows(val_df, step_size=5)
X_test, y_test, test_severity = create_windows(test_df, step_size=5)

# Non-Overlapping Windows (Robustness Experiment)
X_test_nonoverlap, y_test_nonoverlap, test_severity_nonoverlap = create_windows(test_df, step_size=20)

X_train_normal = X_train[y_train == 0]
X_val_normal = X_val[y_val == 0]

print("\n--- WINDOW-LEVEL CLASS DISTRIBUTION (OVERLAPPING) ---")
for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"\n{name}:")
    print(pd.Series(labels).value_counts())
    print(pd.Series(labels).value_counts(normalize=True))


Chronological Split - Train: 3,299,445, Val: 471,349, Test: 942,699 logs.

--- WINDOW-LEVEL CLASS DISTRIBUTION (OVERLAPPING) ---

Train:
0    601357
1     58529
Name: count, dtype: int64
0    0.911304
1    0.088696
Name: proportion, dtype: float64

Validation:
0    83601
1    10665
Name: count, dtype: int64
0    0.886863
1    0.113137
Name: proportion, dtype: float64

Test:
0    176464
1     12072
Name: count, dtype: int64
0    0.93597
1    0.06403
Name: proportion, dtype: float64


In [ ]:
# ==========================================
# CELL 3: DIAGNOSTICS & VOCABULARY DRIFT ANALYSIS
# ==========================================
def print_unk_statistics(name, data, X_data, y_data):
    tokens = data['Event_Token'].values
    overall_unk = np.mean(tokens == 0)
    normal_mask = data['Is_Anomaly'].values == 0
    anomaly_mask = data['Is_Anomaly'].values == 1
    normal_unk = np.mean(tokens[normal_mask] == 0) if normal_mask.sum() else 0
    anomaly_unk = np.mean(tokens[anomaly_mask] == 0) if anomaly_mask.sum() else 0

    unk_per_window = (X_data == 0).sum(axis=1)
    all_unk_windows = (X_data == 0).all(axis=1).mean()
    normal_windows = y_data == 0
    anomaly_windows = y_data == 1

    print(f"\n--- {name} VOCABULARY DRIFT ---")
    print(f"Token-Level OOV   | Overall: {overall_unk:.4%} | Normal: {normal_unk:.4%} | Anomaly: {anomaly_unk:.4%}")
    print(f"Windows with OOV  | Overall: {(unk_per_window > 0).mean():.4%}")
    if normal_windows.any(): print(f"                    Normal:  {(unk_per_window[normal_windows] > 0).mean():.4%}")
    if anomaly_windows.any(): print(f"                    Anomaly: {(unk_per_window[anomaly_windows] > 0).mean():.4%}")
    print(f"All-OOV Windows   | {all_unk_windows:.4%}")

print_unk_statistics("TRAIN", train_df, X_train, y_train)
print_unk_statistics("VALIDATION", val_df, X_val, y_val)
print_unk_statistics("TEST", test_df, X_test, y_test)

normal_val_severity = val_severity[y_val == 0]
print("\n--- NORMAL VALIDATION SEVERITY DISTRIBUTION ---")
print(pd.Series(normal_val_severity).value_counts().sort_index().head(10))


--- TRAIN VOCABULARY DRIFT ---
Token-Level OOV   | Overall: 0.0000% | Normal: 0.0000% | Anomaly: 0.0000%
Windows with OOV  | Overall: 0.0000%
                    Normal:  0.0000%
                    Anomaly: 0.0000%
All-OOV Windows   | 0.0000%

--- VALIDATION VOCABULARY DRIFT ---
Token-Level OOV   | Overall: 26.6830% | Normal: 22.9111% | Anomaly: 61.2592%
Windows with OOV  | Overall: 31.4238%
                    Normal:  27.3789%
                    Anomaly: 63.1317%
All-OOV Windows   | 23.3997%

--- TEST VOCABULARY DRIFT ---
Token-Level OOV   | Overall: 75.4358% | Normal: 75.0870% | Anomaly: 82.1922%
Windows with OOV  | Overall: 80.9283%
                    Normal:  80.4527%
                    Anomaly: 87.8810%
All-OOV Windows   | 70.8300%

--- NORMAL VALIDATION SEVERITY DISTRIBUTION ---
0    49010
1      790
2      225
3      184
4      141
5      107
6      109
7       87
8       92
9      109
Name: count, dtype: int64


In [ ]:
# ==========================================
# CELL 4: LSTM SEQUENCE AUTOENCODER TRAINING
# ==========================================
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

batch_size = 512
vocab_size = len(template_to_id) + 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(TensorDataset(torch.tensor(X_train_normal, dtype=torch.long), torch.tensor(X_train_normal, dtype=torch.long)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_normal, dtype=torch.long), torch.tensor(X_val_normal, dtype=torch.long)), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(X_test, dtype=torch.long)), batch_size=batch_size, shuffle=False)
test_loader_nonoverlap = DataLoader(TensorDataset(torch.tensor(X_test_nonoverlap, dtype=torch.long), torch.tensor(X_test_nonoverlap, dtype=torch.long)), batch_size=batch_size, shuffle=False)

class LogAutoencoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_dim=32):
        super(LogAutoencoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.encoder = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, embedding_dim, batch_first=True)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.encoder(embedded)
        hidden_repeated = hidden.repeat(x.size(1), 1, 1).transpose(0, 1)
        decoded, _ = self.decoder(hidden_repeated)
        return self.output_layer(decoded)

model = LogAutoencoder(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
eval_criterion = nn.CrossEntropyLoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=0.001)

model_save_path = 'best_ae_model.pth'

if os.path.exists(model_save_path):
    print(f"\nFound existing model at '{model_save_path}'. Skipping training and loading weights...")
    model.load_state_dict(torch.load(model_save_path, map_location=device))
else:
    max_epochs = 30
    patience = 3
    min_delta = 1e-4
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []

    print(f"\nStarting Training on {device} with Early Stopping...")
    for epoch in range(max_epochs):
        model.train()
        total_train_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data).view(-1, vocab_size)
            loss = criterion(output, target.view(-1))
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data).view(-1, vocab_size)
                val_loss = criterion(output, target.view(-1))
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{max_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss - min_delta:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at Epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(model_save_path, map_location=device))


Starting Training on cuda with Early Stopping...
Epoch [1/30] | Train Loss: 2.7022 | Val Loss: 5.8699
Epoch [2/30] | Train Loss: 1.2261 | Val Loss: 4.9366
Epoch [3/30] | Train Loss: 0.8918 | Val Loss: 4.7044
Epoch [4/30] | Train Loss: 0.7211 | Val Loss: 4.5314
Epoch [5/30] | Train Loss: 0.6158 | Val Loss: 4.4221
Epoch [6/30] | Train Loss: 0.5587 | Val Loss: 4.5725
Epoch [7/30] | Train Loss: 0.5233 | Val Loss: 4.5503
Epoch [8/30] | Train Loss: 0.4988 | Val Loss: 4.9305
Early stopping triggered at Epoch 8


<All keys matched successfully>

In [ ]:
# ==========================================
# CELL 5: UNSUPERVISED EVALUATION, ROBUSTNESS & ABLATION
# ==========================================
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, f1_score, average_precision_score, roc_auc_score, accuracy_score, precision_score, recall_score

model.eval()
def get_reconstruction_errors(loader):
    losses = []
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data).transpose(1, 2)
            seq_loss = eval_criterion(output, target).mean(dim=1)
            losses.extend(seq_loss.cpu().numpy())
    return np.array(losses)

val_loader_full = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(X_val, dtype=torch.long)), batch_size=batch_size, shuffle=False)
val_losses_all = get_reconstruction_errors(val_loader_full)
test_losses = get_reconstruction_errors(test_loader)
test_losses_nonoverlap = get_reconstruction_errors(test_loader_nonoverlap)

normal_val_losses = val_losses_all[y_val == 0]

# --- 1. FPR-CONSTRAINED SEVERITY BASELINE CALIBRATION ---
print("\n--- BASELINE CALIBRATION (MAX VALIDATION FPR = 0.05) ---")
MAX_VALIDATION_FPR = 0.05
fpr_base_thresh = 20

for thresh in range(21):
    val_preds = (normal_val_severity > thresh).astype(int)
    current_fpr = val_preds.mean()
    if current_fpr <= MAX_VALIDATION_FPR:
        fpr_base_thresh = thresh
        print(f"Calibrated Baseline Threshold: > {fpr_base_thresh} critical events (Validation FPR: {current_fpr:.4f})")
        break

base_preds_fpr = (test_severity > fpr_base_thresh).astype(int)
base_preds_nonoverlap_fpr = (test_severity_nonoverlap > fpr_base_thresh).astype(int)

# --- 2. THRESHOLD SENSITIVITY ANALYSIS (LSTM AUTOENCODER) ---
percentiles = [90, 95, 97.5, 99]
final_percentile = 95

print("\n--- SENSITIVITY ANALYSIS (PRIMARY TEST SET) ---")
print(f"{'Thresh %':<10} | {'Raw Thresh':<12} | {'Predicted':<10} | {'Precision':<10} | {'Recall':<10} | {'F1':<10} | {'FPR':<10} | {'FNR':<10}")
print("-" * 95)

for p in percentiles:
    ae_thresh = np.percentile(normal_val_losses, p)
    ae_preds = (test_losses > ae_thresh).astype(int)
    a_tn, a_fp, a_fn, a_tp = confusion_matrix(y_test, ae_preds).ravel()
    a_prec = a_tp / (a_tp + a_fp) if (a_tp + a_fp) > 0 else 0
    a_rec = a_tp / (a_tp + a_fn) if (a_tp + a_fn) > 0 else 0
    a_f1 = f1_score(y_test, ae_preds)
    a_fpr = a_fp / (a_fp + a_tn) if (a_fp + a_tn) > 0 else 0
    a_fnr = a_fn / (a_fn + a_tp) if (a_fn + a_tp) > 0 else 0
    print(f"{p:<10.1f} | {ae_thresh:<12.6f} | {ae_preds.sum():<10,} | {a_prec:<10.4f} | {a_rec:<10.4f} | {a_f1:<10.4f} | {a_fpr:<10.4f} | {a_fnr:<10.4f}")

    if p == final_percentile:
        final_ae_preds = ae_preds
        final_ae_thresh = ae_thresh
        ae_preds_nonoverlap = (test_losses_nonoverlap > final_ae_thresh).astype(int)

# --- 3. PRIMARY EVALUATION (95TH PERCENTILE vs FPR-CONSTRAINED BASELINE) ---
base_tn, base_fp, base_fn, base_tp = confusion_matrix(y_test, base_preds_fpr).ravel()
ae_tn, ae_fp, ae_fn, ae_tp = confusion_matrix(y_test, final_ae_preds).ravel()

print(f"\n--- PRIMARY EVALUATION (LSTM 95TH PERCENTILE vs. SEVERITY BASELINE FPR<=5%) ---")
print("LSTM Sequence Autoencoder:")
print(f"Accuracy: {accuracy_score(y_test, final_ae_preds):.4f} | Precision: {precision_score(y_test, final_ae_preds):.4f} | Recall: {recall_score(y_test, final_ae_preds):.4f} | F1: {f1_score(y_test, final_ae_preds):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, test_losses):.4f} | ROC-AUC: {roc_auc_score(y_test, test_losses):.4f}")
print(f"TN: {ae_tn:,} | FP: {ae_fp:,} | FN: {ae_fn:,} | TP: {ae_tp:,}")
print(f"FPR: {ae_fp / (ae_fp + ae_tn):.4f} | FNR: {ae_fn / (ae_fn + ae_tp):.4f}")

print("\nSeverity Baseline (FPR Constrained):")
print(f"Accuracy: {accuracy_score(y_test, base_preds_fpr):.4f} | Precision: {precision_score(y_test, base_preds_fpr, zero_division=0):.4f} | Recall: {recall_score(y_test, base_preds_fpr):.4f} | F1: {f1_score(y_test, base_preds_fpr):.4f}")
print(f"TN: {base_tn:,} | FP: {base_fp:,} | FN: {base_fn:,} | TP: {base_tp:,}")
print(f"FPR: {base_fp / (base_fp + base_tn):.4f} | FNR: {base_fn / (base_fn + base_tp):.4f}\n")

# --- 4. ROBUSTNESS EXPERIMENT (NON-OVERLAPPING TEST SET) ---
print("--- ROBUSTNESS EXPERIMENT (NON-OVERLAPPING TEST SET) ---")
print("Non-Overlapping Test Distribution:")
print(pd.Series(y_test_nonoverlap).value_counts(normalize=True).apply(lambda x: f"{x:.4%}"))

base_tn_no, base_fp_no, base_fn_no, base_tp_no = confusion_matrix(y_test_nonoverlap, base_preds_nonoverlap_fpr).ravel()
ae_tn_no, ae_fp_no, ae_fn_no, ae_tp_no = confusion_matrix(y_test_nonoverlap, ae_preds_nonoverlap).ravel()

print("\nLSTM Sequence Autoencoder (Non-Overlapping):")
print(f"Accuracy : {accuracy_score(y_test_nonoverlap, ae_preds_nonoverlap):.4f} | Precision: {precision_score(y_test_nonoverlap, ae_preds_nonoverlap):.4f}")
print(f"Recall   : {recall_score(y_test_nonoverlap, ae_preds_nonoverlap):.4f} | F1       : {f1_score(y_test_nonoverlap, ae_preds_nonoverlap):.4f}")
print(f"FPR      : {ae_fp_no / (ae_fp_no + ae_tn_no):.4f} | FNR: {ae_fn_no / (ae_fn_no + ae_tp_no):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test_nonoverlap, test_losses_nonoverlap):.4f}")

print("\nSeverity Baseline (Non-Overlapping):")
print(f"Accuracy : {accuracy_score(y_test_nonoverlap, base_preds_nonoverlap_fpr):.4f} | Precision: {precision_score(y_test_nonoverlap, base_preds_nonoverlap_fpr, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test_nonoverlap, base_preds_nonoverlap_fpr):.4f} | F1       : {f1_score(y_test_nonoverlap, base_preds_nonoverlap_fpr):.4f}")
print(f"FPR      : {base_fp_no / (base_fp_no + base_tn_no):.4f} | FNR: {base_fn_no / (base_fn_no + base_tp_no):.4f}")

# --- 5. UNKNOWN-TOKEN MASKED ABLATION ---
def get_reconstruction_errors_masked(loader):
    losses = []
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            target = target.to(device)
            output = model(data).transpose(1, 2)
            token_loss = F.cross_entropy(output, target, reduction='none')
            valid_mask = (target != 0)
            masked_loss = (token_loss * valid_mask).sum(dim=1)
            valid_count = valid_mask.sum(dim=1)
            seq_loss = masked_loss / valid_count.clamp(min=1)
            losses.extend(seq_loss.cpu().numpy())
    return np.array(losses)

print("\n--- UNKNOWN-TOKEN MASKED ABLATION ---")
ablation_val_losses = get_reconstruction_errors_masked(val_loader_full)
ablation_test_losses = get_reconstruction_errors_masked(test_loader)
ablation_thresh = np.percentile(ablation_val_losses[y_val == 0], final_percentile)
ablation_preds = (ablation_test_losses > ablation_thresh).astype(int)

print(f"F1-Score (Standard Reconstruction Loss): {f1_score(y_test, final_ae_preds):.4f}")
print(f"F1-Score (<UNK>-Excluded Masked Mean):   {f1_score(y_test, ablation_preds):.4f}")


--- BASELINE CALIBRATION (MAX VALIDATION FPR = 0.05) ---
Calibrated Baseline Threshold: > 20 critical events (Validation FPR: 0.0000)

--- SENSITIVITY ANALYSIS (PRIMARY TEST SET) ---
Thresh %   | Raw Thresh   | Predicted  | Precision  | Recall     | F1         | FPR        | FNR       
-----------------------------------------------------------------------------------------------
90.0       | 14.221605    | 3,283      | 0.4590     | 0.1248     | 0.1963     | 0.0101     | 0.8752    
95.0       | 14.221605    | 3,283      | 0.4590     | 0.1248     | 0.1963     | 0.0101     | 0.8752    
97.5       | 14.221605    | 3,283      | 0.4590     | 0.1248     | 0.1963     | 0.0101     | 0.8752    
99.0       | 14.221605    | 3,283      | 0.4590     | 0.1248     | 0.1963     | 0.0101     | 0.8752    

--- PRIMARY EVALUATION (LSTM 95TH PERCENTILE vs. SEVERITY BASELINE FPR<=5%) ---
LSTM Sequence Autoencoder:
Accuracy: 0.9345 | Precision: 0.4590 | Recall: 0.1248 | F1: 0.1963
PR-AUC: 0.1809 | ROC-AUC:

In [ ]:
# ==========================================
# CELL 6: THRESHOLD DIAGNOSTICS & SUPERVISED CALIBRATION
# ==========================================
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# 1. AE VALIDATION LOSS DISTRIBUTION DIAGNOSTIC
# ============================================================
print("\n--- AE VALIDATION LOSS DISTRIBUTION DIAGNOSTIC ---")

print(f"Number of normal validation windows: {len(normal_val_losses):,}")
print(
    f"Unique reconstruction-loss values: "
    f"{np.unique(normal_val_losses).size:,}"
)

for p in [90, 95, 97.5, 99]:
    q = np.percentile(normal_val_losses, p)
    count = np.sum(np.isclose(normal_val_losses, q, atol=1e-8))

    print(
        f"{p:>4.1f}th percentile = {q:.6f} | "
        f"windows at threshold ≈ {count:,}"
    )

# All-UNK diagnostics
val_normal_X = X_val[y_val == 0]
all_unk_val = (val_normal_X == 0).all(axis=1)

print(
    f"\nNormal validation windows that are entirely <UNK>: "
    f"{all_unk_val.mean():.4%}"
)

if all_unk_val.sum() > 0:
    print(
        f"Mean AE loss for all-<UNK> normal windows: "
        f"{normal_val_losses[all_unk_val].mean():.6f}"
    )

if (~all_unk_val).sum() > 0:
    print(
        f"Mean AE loss for normal windows with known tokens: "
        f"{normal_val_losses[~all_unk_val].mean():.6f}"
    )

# ============================================================
# 2. EXACT AE VALIDATION-F1 CALIBRATION
# ============================================================
print("\n--- SUPERVISED CALIBRATION (VALIDATION-OPTIMIZED F1) ---")

scores = np.asarray(val_losses_all)
labels = np.asarray(y_val)

# Sort reconstruction scores from highest to lowest
order = np.argsort(-scores)
sorted_scores = scores[order]
sorted_labels = labels[order]

# Cumulative TP / FP up to each position
cum_tp = np.cumsum(sorted_labels)
cum_fp = np.cumsum(1 - sorted_labels)

total_positive = labels.sum()

# First index of every distinct score
group_starts = np.r_[
    0,
    np.where(np.diff(sorted_scores) != 0)[0] + 1
]

# For threshold = sorted_scores[group_start],
# prediction is (score > threshold), so the entire
# current equal-score group must remain NEGATIVE.

tp_before = np.where(
    group_starts > 0,
    cum_tp[group_starts - 1],
    0
)

fp_before = np.where(
    group_starts > 0,
    cum_fp[group_starts - 1],
    0
)

fn_before = total_positive - tp_before

f1_values = (
    2 * tp_before /
    np.maximum(
        2 * tp_before + fp_before + fn_before,
        1
    )
)

best_group = np.argmax(f1_values)

best_ae_thresh = sorted_scores[group_starts[best_group]]
best_ae_val_f1 = f1_values[best_group]

# Freeze threshold before touching test labels
ae_valopt_test_preds = (
    test_losses > best_ae_thresh
).astype(int)

# ============================================================
# 3. VALIDATION-OPTIMIZED SEVERITY BASELINE
# ============================================================
best_base_val_f1 = -1.0
opt_base_thresh = 0

for thresh in range(21):

    val_preds = (val_severity > thresh).astype(int)

    score = f1_score(
        y_val,
        val_preds
    )

    if score > best_base_val_f1:
        best_base_val_f1 = score
        opt_base_thresh = thresh

base_preds_opt = (
    test_severity > opt_base_thresh
).astype(int)

# ============================================================
# 4. FULL TEST METRICS — AE
# ============================================================
a_tn, a_fp, a_fn, a_tp = confusion_matrix(
    y_test,
    ae_valopt_test_preds
).ravel()

a_fpr = (
    a_fp / (a_fp + a_tn)
    if (a_fp + a_tn) > 0 else 0
)

a_fnr = (
    a_fn / (a_fn + a_tp)
    if (a_fn + a_tp) > 0 else 0
)

print("\nLSTM Sequence Autoencoder (Validation-Optimized):")

print(
    f"Selected Threshold: {best_ae_thresh:.6f} | "
    f"Val F1: {best_ae_val_f1:.4f}"
)

print(
    f"Test predicted anomalies: "
    f"{ae_valopt_test_preds.sum():,}"
)

print(
    f"Accuracy : "
    f"{accuracy_score(y_test, ae_valopt_test_preds):.4f} | "
    f"Precision: "
    f"{precision_score(y_test, ae_valopt_test_preds, zero_division=0):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test, ae_valopt_test_preds):.4f} | "
    f"F1       : "
    f"{f1_score(y_test, ae_valopt_test_preds):.4f}"
)

print(
    f"TN: {a_tn:,} | "
    f"FP: {a_fp:,} | "
    f"FN: {a_fn:,} | "
    f"TP: {a_tp:,}"
)

print(
    f"FPR: {a_fpr:.4f} | "
    f"FNR: {a_fnr:.4f}"
)

# ============================================================
# 5. FULL TEST METRICS — SEVERITY BASELINE
# ============================================================
b_tn, b_fp, b_fn, b_tp = confusion_matrix(
    y_test,
    base_preds_opt
).ravel()

b_fpr = (
    b_fp / (b_fp + b_tn)
    if (b_fp + b_tn) > 0 else 0
)

b_fnr = (
    b_fn / (b_fn + b_tp)
    if (b_fn + b_tp) > 0 else 0
)

print("\nSeverity Baseline (Validation-Optimized):")

print(
    f"Selected Threshold: > {opt_base_thresh} "
    f"critical events | Val F1: {best_base_val_f1:.4f}"
)

print(
    f"Test predicted anomalies: "
    f"{base_preds_opt.sum():,}"
)

print(
    f"Accuracy : "
    f"{accuracy_score(y_test, base_preds_opt):.4f} | "
    f"Precision: "
    f"{precision_score(y_test, base_preds_opt, zero_division=0):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test, base_preds_opt):.4f} | "
    f"F1       : "
    f"{f1_score(y_test, base_preds_opt):.4f}"
)

print(
    f"TN: {b_tn:,} | "
    f"FP: {b_fp:,} | "
    f"FN: {b_fn:,} | "
    f"TP: {b_tp:,}"
)

print(
    f"FPR: {b_fpr:.4f} | "
    f"FNR: {b_fnr:.4f}"
)


--- AE VALIDATION LOSS DISTRIBUTION DIAGNOSTIC ---
Number of normal validation windows: 83,601
Unique reconstruction-loss values: 27,928
90.0th percentile = 14.221605 | windows at threshold ≈ 16,802
95.0th percentile = 14.221605 | windows at threshold ≈ 16,802
97.5th percentile = 14.221605 | windows at threshold ≈ 16,802
99.0th percentile = 14.221605 | windows at threshold ≈ 16,802

Normal validation windows that are entirely <UNK>: 20.0978%
Mean AE loss for all-<UNK> normal windows: 14.221604
Mean AE loss for normal windows with known tokens: 1.903386

--- SUPERVISED CALIBRATION (VALIDATION-OPTIMIZED F1) ---

LSTM Sequence Autoencoder (Validation-Optimized):
Selected Threshold: 11.430791 | Val F1: 0.4906
Test predicted anomalies: 142,433
Accuracy : 0.3022 | Precision: 0.0805
Recall   : 0.9500 | F1       : 0.1485
TN: 45,500 | FP: 130,964 | FN: 603 | TP: 11,469
FPR: 0.7422 | FNR: 0.0500

Severity Baseline (Validation-Optimized):
Selected Threshold: > 1 critical events | Val F1: 0.3842


In [ ]:
# ============================================================
# 6. THRESHOLD-INDEPENDENT SCORE COMPARISON
# ============================================================
from sklearn.metrics import average_precision_score, roc_auc_score

print("\n--- THRESHOLD-INDEPENDENT SCORE COMPARISON ---")

severity_val_pr_auc = average_precision_score(y_val, val_severity)
severity_test_pr_auc = average_precision_score(y_test, test_severity)
severity_val_roc_auc = roc_auc_score(y_val, val_severity)
severity_test_roc_auc = roc_auc_score(y_test, test_severity)

print(f"Severity Validation PR-AUC : {severity_val_pr_auc:.4f}")
print(f"Severity Test PR-AUC       : {severity_test_pr_auc:.4f}")
print(f"Severity Validation ROC-AUC: {severity_val_roc_auc:.4f}")
print(f"Severity Test ROC-AUC      : {severity_test_roc_auc:.4f}")

print(f"\nLSTM AE Test PR-AUC        : {average_precision_score(y_test, test_losses):.4f}")
print(f"LSTM AE Test ROC-AUC       : {roc_auc_score(y_test, test_losses):.4f}")


--- THRESHOLD-INDEPENDENT SCORE COMPARISON ---
Severity Validation PR-AUC : 0.2284
Severity Test PR-AUC       : 0.4907
Severity Validation ROC-AUC: 0.7853
Severity Test ROC-AUC      : 0.9660

LSTM AE Test PR-AUC        : 0.1809
LSTM AE Test ROC-AUC       : 0.6146
